In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yfinance as yf
import talib
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
import torch.optim as optim
import os
from sklearn.model_selection import TimeSeriesSplit

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss, mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt

import optuna
from optuna.samplers import TPESampler
from optuna.trial import TrialState
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau 
import shap
import plotly.graph_objs as go
import plotly.offline as pyo
from tqdm.auto import tqdm
from sklearn.utils.class_weight import compute_class_weight



In [2]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("gpu")
else:
    device = torch.device('cpu')
print(torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)
print('cuDNN version:', torch.backends.cudnn.version())

gpu
2.1.2+cu121
CUDA available: True
CUDA version: 12.1
cuDNN version: 8902


In [3]:
start_date = '2018-01-01'
end_date = '2024-01-24'

stock_data = yf.download("AAPL", start=start_date, end=end_date)[["Adj Close", "High", "Low", "Volume"]]

stock_data = stock_data.reset_index()

stock_data = stock_data[['Date', 'Adj Close', "High", "Low", "Volume"]]

stock_data = stock_data.sort_values(by="Date")
stock_data.head(45)

[*********************100%%**********************]  1 of 1 completed


,Date,Adj Close,High,Low,Volume
0,2018-01-02,40.722874,43.075001,42.314999,102223600
1,2018-01-03,40.715790,43.637501,42.990002,118071600
2,2018-01-04,40.904911,43.367500,43.020000,89738400
3,2018-01-05,41.370613,43.842499,43.262501,94640000
4,2018-01-08,41.216961,43.902500,43.482498,82271200
5,2018-01-09,41.212223,43.764999,43.352501,86336000
6,2018-01-10,41.202774,43.575001,43.250000,95839600
7,2018-01-11,41.436825,43.872501,43.622501,74670800
8,2018-01-12,41.864700,44.340000,43.912498,101672400
9,2018-01-16,41.651943,44.847500,44.035000,118263600


In [4]:
time_step = 44

In [5]:
test_index = int((len(stock_data)-44)*0.8+44+44)

In [6]:
date = stock_data["Date"].iloc[time_step:].dt.strftime('%Y-%m-%d')
date_test = stock_data["Date"].iloc[test_index:].reset_index()
date_test.drop(columns=["index"], inplace=True)
date_test

,Date
0,2023-01-23
1,2023-01-24
2,2023-01-25
3,2023-01-26
4,2023-01-27
...,...
247,2024-01-17
248,2024-01-18
249,2024-01-19
250,2024-01-22


In [7]:
def add_technical_indicators(data, timeperiod=time_step):

    # MACD
    macd, macdsignal, macdhist = talib.MACD(data["Adj Close"], fastperiod=12, slowperiod=26, signalperiod=9)

    rsi = talib.RSI(data["Adj Close"], timeperiod=14)

    # CMO
    cmo = talib.CMO(data["Adj Close"], timeperiod=timeperiod)

    # MOM
    mom = talib.MOM(data["Adj Close"], timeperiod=timeperiod)

    # Bollinger Bands
    upperband, middleband, lowerband = talib.BBANDS(data["Adj Close"], timeperiod=20, nbdevup=2, nbdevdn=2, matype=0)

    # SMA
    sma_s = talib.SMA(data["Adj Close"], timeperiod=20)
    sma_l = talib.SMA(data["Adj Close"], timeperiod=50)

    # Calculate Exponential Moving Average (EMA)
    ema = talib.EMA(data["Adj Close"], timeperiod=timeperiod)

    # Calculate Stochastic Oscillator
    slowk, slowd = talib.STOCH(data['High'], data['Low'], data['Adj Close'], fastk_period=14, slowk_period=3, slowk_matype=0, slowd_period=3, slowd_matype=0)

    # Calculate Average True Range (ATR)
    atr = talib.ATR(data['High'], data['Low'], data['Adj Close'], timeperiod=14)

    # Calculate On-Balance Volume (OBV)
    obv = talib.OBV(data['Adj Close'], data['Volume'])

    # Combine all indicators into a DataFrame
    indicators = pd.DataFrame({
        'MACD': macd,
        'MACD_signal': macdsignal,
        'RSI': rsi,
        'CMO': cmo,
        'MOM': mom,
        'Upper_BB': upperband,
        'Middle_BB': middleband,
        'Lower_BB': lowerband,
        'SMA_SHORT': sma_s,
        'SMA_LONG': sma_l,
        'EMA': ema,
        'SLOWK': slowk,
        'SLOWD': slowd,
        'ATR': atr,
        'OBV': obv,

    })
    return indicators

In [8]:
indicators = add_technical_indicators(stock_data)
indicators.head(45)

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102223600.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-15848000.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73890400.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,168530400.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86259200.0
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-76800.0
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-95916400.0
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-21245600.0
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80426800.0
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-37836800.0


In [9]:
indicators_with_price = pd.concat([indicators, stock_data["Adj Close"]], axis=1, join='inner')
indicators_with_price.head(45)

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102223600.0,40.722874
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-15848000.0,40.715790
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73890400.0,40.904911
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,168530400.0,41.370613
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86259200.0,41.216961
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-76800.0,41.212223
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-95916400.0,41.202774
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-21245600.0,41.436825
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80426800.0,41.864700
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-37836800.0,41.651943


In [10]:
indicators_with_price = indicators_with_price.dropna()
indicators_with_price = indicators_with_price.reset_index(drop=True)
indicators_with_price

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close
0,0.567014,0.443929,58.241601,7.100880,1.143620,43.382888,41.728832,40.074776,41.728832,40.814298,41.035721,-7.013324,-3.909598,2.715223,-3.124152e+08,42.355843
1,0.548494,0.464842,58.592114,7.332854,1.202904,43.261033,41.862707,40.464380,41.862707,40.847954,41.096608,-20.016700,-8.448344,2.714432,-2.214400e+08,42.405678
2,0.515806,0.475035,57.044942,6.516263,0.819328,43.280291,41.922405,40.564519,41.922405,40.878762,41.148143,-27.991838,-18.340621,2.690139,-3.790588e+08,42.256153
3,0.432813,0.466591,50.806392,3.052132,-0.254192,43.245421,41.956468,40.667514,41.956468,40.892874,41.168693,-36.985315,-28.331285,2.648797,-5.128460e+08,41.610508
4,0.361721,0.445617,50.674723,2.976529,-0.055679,43.183919,41.996702,40.809484,41.996702,40.897387,41.187696,-46.751996,-37.243050,2.644561,-5.914436e+08,41.596264
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1470,-1.918426,-1.197745,36.736038,-3.771153,-3.720001,199.131030,188.796999,178.462968,188.796999,189.292038,187.549134,26.310155,30.750307,3.101382,3.547580e+09,182.679993
1471,-1.555397,-1.269276,51.604604,4.598480,3.830002,198.242596,188.434000,178.625403,188.434000,189.536287,187.597173,33.195313,30.241850,3.341284,3.625586e+09,188.630005
1472,-1.019515,-1.219324,56.967975,8.324259,4.119995,197.297528,188.164999,179.032471,188.164999,189.787603,187.773298,51.916516,37.140661,3.339763,3.694327e+09,191.559998
1473,-0.402178,-1.055894,60.698088,11.147862,5.880005,197.121605,188.117999,179.114394,188.117999,190.033787,188.045152,76.309536,53.807122,3.370495,3.754460e+09,193.889999


In [11]:
# Define the conditions
conditions = [
    (indicators_with_price['RSI'] < 30), # al sinyali
    (indicators_with_price['RSI'] > 70) # sat sinyali
]

# Define the choices corresponding to each condition
choices = [2, 1] # 1 sat demek 2 al

# Use np.select to apply the conditions and choices
# The default value is 0 (for when the RSI is between 30 and 70)
indicators_with_price['RSI_SIGNAL'] = np.select(conditions, choices, default=0)

conditions = [
    (indicators_with_price['SMA_SHORT'] > indicators_with_price['SMA_LONG']), # al sinyali
    (indicators_with_price['SMA_LONG'] > indicators_with_price['SMA_SHORT']) # sat sinyali
]
# Define the choices corresponding to each condition
choices = [2, 1] # 1 sat demek 2 al

indicators_with_price['SMA_SIGNAL'] = np.select(conditions, choices, default=0)

# Display the dataframe to verify the results

# Create a column 'MACD_Signal' to hold the trading signal
indicators_with_price['MACD_Signal'] = 0

# Generate buy/sell signals
indicators_with_price['MACD_Signal'][indicators_with_price['MACD'] > indicators_with_price['MACD_signal']] = 2  # Buy signal
indicators_with_price['MACD_Signal'][indicators_with_price['MACD'] < indicators_with_price['MACD_signal']] = 1  # Sell signal

# Create a column 'BB_Signal' to hold the trading signal
indicators_with_price['BB_Signal'] = 0

# Generate buy/sell signals
indicators_with_price['BB_Signal'][indicators_with_price['Adj Close'] < indicators_with_price['Lower_BB']] = 2  # Buy signal
indicators_with_price['BB_Signal'][indicators_with_price['Adj Close'] > indicators_with_price['Upper_BB']] = 1  # Sell signal



indicators_with_price.head(50)

/tmp/ipykernel_18050/840148971.py:29: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_18050/840148971.py:30: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_18050/840148971.py:36: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_18050/840148971.py:37: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pyd

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close,RSI_SIGNAL,SMA_SIGNAL,MACD_Signal,BB_Signal
0,0.567014,0.443929,58.241601,7.100880,1.143620,43.382888,41.728832,40.074776,41.728832,40.814298,41.035721,-7.013324,-3.909598,2.715223,-3.124152e+08,42.355843,0,2,2,0
1,0.548494,0.464842,58.592114,7.332854,1.202904,43.261033,41.862707,40.464380,41.862707,40.847954,41.096608,-20.016700,-8.448344,2.714432,-2.214400e+08,42.405678,0,2,2,0
2,0.515806,0.475035,57.044942,6.516263,0.819328,43.280291,41.922405,40.564519,41.922405,40.878762,41.148143,-27.991838,-18.340621,2.690139,-3.790588e+08,42.256153,0,2,2,0
3,0.432813,0.466591,50.806392,3.052132,-0.254192,43.245421,41.956468,40.667514,41.956468,40.892874,41.168693,-36.985315,-28.331285,2.648797,-5.128460e+08,41.610508,0,2,1,0
4,0.361721,0.445617,50.674723,2.976529,-0.055679,43.183919,41.996702,40.809484,41.996702,40.897387,41.187696,-46.751996,-37.243050,2.644561,-5.914436e+08,41.596264,0,2,1,0
5,0.226727,0.401839,42.776436,-1.895784,-1.685963,43.175302,41.999076,40.822850,41.999076,40.886126,41.163972,-59.960257,-47.899189,2.611109,-7.396632e+08,40.653912,0,2,1,2
6,0.072556,0.335982,38.805945,-4.708055,-2.298210,43.330936,41.955756,40.580576,41.955756,40.863471,41.115773,-60.364790,-55.692348,2.604321,-9.056264e+08,40.079487,0,2,1,2
7,-0.123098,0.244166,33.410012,-9.019923,-3.037212,43.669846,41.830427,39.991008,41.830427,40.822443,41.028466,-57.037896,-59.120981,2.589764,-1.069742e+09,39.151379,0,2,1,2
8,-0.126721,0.169989,48.771961,0.230800,-0.833458,43.603899,41.756843,39.909787,41.756843,40.813906,41.027644,-35.113242,-50.838643,2.699325,-9.195768e+08,41.009972,0,2,1,0
9,-0.211999,0.093591,42.761393,-4.462410,-1.894447,43.620650,41.637566,39.654482,41.637566,40.775781,40.980124,-25.755897,-39.302345,2.704911,-1.083267e+09,39.958431,0,2,1,0


In [12]:
# Sum the signals for each row
indicators_with_price['Sum_Buy_Signals'] = (indicators_with_price[['RSI_SIGNAL', 'SMA_SIGNAL', 'MACD_Signal', 'BB_Signal']] == 2).sum(axis=1)
indicators_with_price['Sum_Sell_Signals'] = (indicators_with_price[['RSI_SIGNAL', 'SMA_SIGNAL', 'MACD_Signal', 'BB_Signal']] == 1).sum(axis=1)

# Define the conditions
conditions = [
    (indicators_with_price['Sum_Buy_Signals'] >= 3), # At least three buy signals - Strong Buy
    (indicators_with_price['Sum_Buy_Signals'] >= 2), # At least two buy signals - Buy
    (indicators_with_price['Sum_Sell_Signals'] >= 3), # At least three sell signals - Strong Sell
    (indicators_with_price['Sum_Sell_Signals'] >= 2), # At least two sell signals - Sell
]

# Define the choices corresponding to each condition
choices = [4, 3, 1, 2] # Strong Buy, Buy, Strong Sell, Sell

# Use np.select to apply the conditions and choices
# The default value is 0 (Neutral)
indicators_with_price['Signal'] = np.select(conditions, choices, default=0)

# Remove the sum columns if they are no longer needed
indicators_with_price = indicators_with_price.drop(columns=['Sum_Buy_Signals', 'Sum_Sell_Signals'])

# Display the dataframe to verify the results
indicators_with_price[['RSI_SIGNAL', 'SMA_SIGNAL', 'MACD_Signal', 'BB_Signal', 'Signal']].head(50)

,RSI_SIGNAL,SMA_SIGNAL,MACD_Signal,BB_Signal,Signal
0,0,2,2,0,3
1,0,2,2,0,3
2,0,2,2,0,3
3,0,2,1,0,0
4,0,2,1,0,0
5,0,2,1,2,3
6,0,2,1,2,3
7,0,2,1,2,3
8,0,2,1,0,0
9,0,2,1,0,0


In [13]:
signal_distribution = indicators_with_price['Signal'].value_counts().sort_index()
signal_distribution

Signal
0    697
1      4
2    263
3    504
4      7
Name: count, dtype: int64

In [14]:
# indicators_with_price['Next_Adj_Close'] = indicators_with_price['Adj Close'].shift(-1)
# indicators_with_price['Return'] = ((indicators_with_price['Next_Adj_Close'] - indicators_with_price['Adj Close'])/indicators_with_price['Adj Close'])*100
# indicators_with_price['Signal'] = np.where(indicators_with_price['Return'] > 1, 1,
#                                            np.where(indicators_with_price['Return'] < -1, 2, 0))
# indicators_with_price


In [15]:
indicators_with_price["Signal"].value_counts()

Signal
0    697
3    504
2    263
4      7
1      4
Name: count, dtype: int64

In [16]:
indicators_with_price.columns

Index(['MACD', 'MACD_signal', 'RSI', 'CMO', 'MOM', 'Upper_BB', 'Middle_BB',
       'Lower_BB', 'SMA_SHORT', 'SMA_LONG', 'EMA', 'SLOWK', 'SLOWD', 'ATR',
       'OBV', 'Adj Close', 'RSI_SIGNAL', 'SMA_SIGNAL', 'MACD_Signal',
       'BB_Signal', 'Signal'],
      dtype='object')

In [17]:
# indicators_with_price = indicators_with_price.drop(columns=['Next_Adj_Close', 'Return'])
# indicators_with_price

In [18]:
indicators_with_price.drop(columns=['RSI_SIGNAL', 'SMA_SIGNAL', 'MACD_Signal', 'BB_Signal'], inplace=True)
indicators_with_price.head(50)

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close,Signal
0,0.567014,0.443929,58.241601,7.100880,1.143620,43.382888,41.728832,40.074776,41.728832,40.814298,41.035721,-7.013324,-3.909598,2.715223,-3.124152e+08,42.355843,3
1,0.548494,0.464842,58.592114,7.332854,1.202904,43.261033,41.862707,40.464380,41.862707,40.847954,41.096608,-20.016700,-8.448344,2.714432,-2.214400e+08,42.405678,3
2,0.515806,0.475035,57.044942,6.516263,0.819328,43.280291,41.922405,40.564519,41.922405,40.878762,41.148143,-27.991838,-18.340621,2.690139,-3.790588e+08,42.256153,3
3,0.432813,0.466591,50.806392,3.052132,-0.254192,43.245421,41.956468,40.667514,41.956468,40.892874,41.168693,-36.985315,-28.331285,2.648797,-5.128460e+08,41.610508,0
4,0.361721,0.445617,50.674723,2.976529,-0.055679,43.183919,41.996702,40.809484,41.996702,40.897387,41.187696,-46.751996,-37.243050,2.644561,-5.914436e+08,41.596264,0
5,0.226727,0.401839,42.776436,-1.895784,-1.685963,43.175302,41.999076,40.822850,41.999076,40.886126,41.163972,-59.960257,-47.899189,2.611109,-7.396632e+08,40.653912,3
6,0.072556,0.335982,38.805945,-4.708055,-2.298210,43.330936,41.955756,40.580576,41.955756,40.863471,41.115773,-60.364790,-55.692348,2.604321,-9.056264e+08,40.079487,3
7,-0.123098,0.244166,33.410012,-9.019923,-3.037212,43.669846,41.830427,39.991008,41.830427,40.822443,41.028466,-57.037896,-59.120981,2.589764,-1.069742e+09,39.151379,3
8,-0.126721,0.169989,48.771961,0.230800,-0.833458,43.603899,41.756843,39.909787,41.756843,40.813906,41.027644,-35.113242,-50.838643,2.699325,-9.195768e+08,41.009972,0
9,-0.211999,0.093591,42.761393,-4.462410,-1.894447,43.620650,41.637566,39.654482,41.637566,40.775781,40.980124,-25.755897,-39.302345,2.704911,-1.083267e+09,39.958431,0


In [19]:
# y = indicators_with_price["Adj Close"]
# y_2 = indicators_with_price["SMA"]
# y_3 = indicators_with_price["EMA"]
# y_4 = indicators_with_price["Upper_BB"]
# y_5 = indicators_with_price["Middle_BB"]
# y_6 = indicators_with_price["Lower_BB"]
# X = np.array(date)

# trace = go.Scatter(x=X, y=y, mode="lines", name="Adj Close")
# trace_2 = go.Scatter(x=X, y=y_2, mode="lines", name="SMA")
# trace_3 = go.Scatter(x=X, y=y_3, mode="lines", name="EMA")
# trace_4 = go.Scatter(x=X, y=y_4, mode="lines", name="Upper_BB")
# trace_5 = go.Scatter(x=X, y=y_5, mode="lines", name="Middle_BB")
# trace_6 = go.Scatter(x=X, y=y_6, mode="lines", name="Lower_BB")



# layout = go.Layout(
#     title='Stock Price and Volume',
#     xaxis=dict(title='Date'),
#     yaxis=dict(title='Adj Close', side='left', rangemode='tozero'),
#     yaxis2=dict(title='SMA', side='right', overlaying='y', rangemode='tozero'),
#     yaxis3=dict(title='EMA', side='right', overlaying='y', rangemode='tozero'),
#     yaxis4=dict(title='Upper_BB', side='right', overlaying='y', rangemode='tozero'),
#     yaxis5=dict(title='Middle_BB', side='right', overlaying='y', rangemode='tozero'),
#     yaxis6=dict(title='Lower_BB', side='right', overlaying='y', rangemode='tozero'),
#     height=600,
# )

# fig = go.Figure(data=[trace, trace_2, trace_3, trace_4, trace_5, trace_6], layout=layout)

# # Show plot
# pyo.iplot(fig)

In [20]:
class RollingWindowDataset(Dataset):
    def __init__(self, X, y, window_size, device="gpu"):
        self.X = X.clone().detach().to(torch.float).to(device)
        self.y = y.clone().detach().to(torch.float).to(device)
        self.window_size = window_size

    def __len__(self):
        # Adjust the length to account for window size
        return len(self.X) - self.window_size 

    def __getitem__(self, idx):
        # Ensure idx is within the valid range
        if idx + self.window_size > len(self.X):
            raise IndexError("Index out of bounds")

        # Extract a window of data from X
        X_window = self.X[idx : idx + self.window_size]
        
        # Assuming you're predicting the value right after the window
        y_target = self.y[idx + self.window_size]  

        return X_window.clone().detach().to(torch.float), y_target.clone().detach().to(torch.float)


In [21]:
X = indicators_with_price.iloc[:, :-1]
y = indicators_with_price.iloc[:,-1]

signal_true = indicators_with_price.iloc[:,-1]
y

0       3
1       3
2       3
3       0
4       0
       ..
1470    2
1471    2
1472    0
1473    0
1474    0
Name: Signal, Length: 1475, dtype: int64

In [22]:
train_signal_true = signal_true.iloc[:int(len(X)*0.8)]
test_signal_true = signal_true.iloc[int(len(X)*0.8):]
test_signal_true

1180    0
1181    0
1182    3
1183    3
1184    3
       ..
1470    2
1471    2
1472    0
1473    0
1474    0
Name: Signal, Length: 295, dtype: int64

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
# X_train = X_train.to_numpy()
# y_train = y_train.to_numpy()
y_test.head(44)

1180    0
1181    0
1182    3
1183    3
1184    3
1185    3
1186    3
1187    2
1188    2
1189    0
1190    0
1191    0
1192    0
1193    0
1194    0
1195    0
1196    0
1197    0
1198    0
1199    3
1200    3
1201    2
1202    2
1203    2
1204    2
1205    2
1206    2
1207    2
1208    2
1209    2
1210    2
1211    2
1212    2
1213    2
1214    0
1215    0
1216    0
1217    0
1218    0
1219    0
1220    0
1221    0
1222    0
1223    2
Name: Signal, dtype: int64

In [24]:
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
class_weights[4]

59.0

In [25]:
# # Get the count of each unique value in the series
# category_counts = y_train.value_counts()
# print(category_counts.index)
# print(category_counts.values)

# # Create bar plot
# plt.bar(["1","0"], category_counts.values)

# # Adding labels and title
# plt.xlabel('Category')
# plt.ylabel('Frequency')
# plt.title('Distribution of Categories')

# # Show plot
# plt.show()

In [26]:
X_test

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close
1180,0.423526,-0.563852,55.445392,0.846042,0.268082,156.691698,145.708436,134.725174,145.708436,146.191966,146.545681,65.483628,62.136599,5.360574,2.756478e+09,149.882217
1181,0.755432,-0.299995,56.065708,1.304430,-2.917862,157.096172,145.920445,134.744718,145.920445,146.076225,146.719165,69.529924,64.941426,5.178945,2.831307e+09,150.449066
1182,0.746664,-0.090663,51.612740,-1.380048,-8.582184,156.996709,145.861152,134.725595,145.861152,145.774922,146.739971,72.169962,69.061171,5.003954,2.772583e+09,147.187286
1183,0.903428,0.108155,54.204203,0.406558,-3.267166,156.748142,145.766296,134.784450,145.766296,145.707196,146.855759,73.612770,71.770885,4.895815,2.824387e+09,149.345215
1184,1.086560,0.303836,55.262544,1.137776,-1.409149,156.967746,145.864117,134.760489,145.864117,145.627983,147.005740,75.694642,73.825791,4.723971,2.882688e+09,150.230316
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1470,-1.918426,-1.197745,36.736038,-3.771153,-3.720001,199.131030,188.796999,178.462968,188.796999,189.292038,187.549134,26.310155,30.750307,3.101382,3.547580e+09,182.679993
1471,-1.555397,-1.269276,51.604604,4.598480,3.830002,198.242596,188.434000,178.625403,188.434000,189.536287,187.597173,33.195313,30.241850,3.341284,3.625586e+09,188.630005
1472,-1.019515,-1.219324,56.967975,8.324259,4.119995,197.297528,188.164999,179.032471,188.164999,189.787603,187.773298,51.916516,37.140661,3.339763,3.694327e+09,191.559998
1473,-0.402178,-1.055894,60.698088,11.147862,5.880005,197.121605,188.117999,179.114394,188.117999,190.033787,188.045152,76.309536,53.807122,3.370495,3.754460e+09,193.889999


In [27]:
X_train_df = pd.DataFrame()
X_test_df = pd.DataFrame()
scaler_dict = {}

X_train_df = X_train
X_test_df = X_test

for column in X_train.columns:

    if column not in ["RSI", "Upper_BB", "Lower_BB", "Middle_BB", "SMA_SHORT", "SMA_LONG", "Return"]:
        scaler = MinMaxScaler()

        # Fit on the training data and transform it
        X_train_scaled = scaler.fit_transform(X_train[[column]].values)
        X_train_df[column] = X_train_scaled
            
        # Transform the test data using the same scaler
        X_test_scaled = scaler.transform(X_test[[column]].values)
        X_test_df[column] = X_test_scaled

        scaler_dict[column] = scaler


X_train_df.head(24)

features = X_train_df.columns
X_train_df

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close
0,0.510132,0.491711,58.241601,0.465351,0.498238,43.382888,41.728832,40.074776,41.728832,40.814298,0.013598,0.543591,0.529471,0.178111,0.245984,0.056481
1,0.508804,0.493383,58.592114,0.468236,0.498960,43.261033,41.862707,40.464380,41.862707,40.847954,0.014072,0.483866,0.507174,0.177936,0.261686,0.056823
2,0.506461,0.494198,57.044942,0.458080,0.494286,43.280291,41.922405,40.564519,41.922405,40.878762,0.014472,0.447235,0.458579,0.172548,0.234481,0.055798
3,0.500513,0.493523,50.806392,0.414999,0.481205,43.245421,41.956468,40.667514,41.956468,40.892874,0.014632,0.405928,0.409500,0.163379,0.211390,0.051371
4,0.495417,0.491846,50.674723,0.414059,0.483624,43.183919,41.996702,40.809484,41.996702,40.897387,0.014780,0.361069,0.365721,0.162439,0.197825,0.051273
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1175,0.340107,0.343542,52.078571,0.349552,0.360230,154.868504,143.872760,132.877016,143.872760,146.786650,0.829435,0.680158,0.631199,0.850805,0.753803,0.767544
1176,0.388138,0.347930,54.750766,0.377027,0.321213,155.224487,144.447009,133.669532,144.447009,146.627571,0.830427,0.747328,0.665570,0.849088,0.770024,0.786842
1177,0.418622,0.358240,53.149272,0.363377,0.420018,155.507896,144.750603,133.993309,144.750603,146.482940,0.830887,0.835637,0.739664,0.803296,0.757360,0.777158
1178,0.453092,0.374178,54.909711,0.380639,0.423561,155.958144,145.075190,134.192236,145.075190,146.398729,0.831932,0.856512,0.802536,0.812803,0.772871,0.789160


In [28]:

# BB bands
scaler_bb = MinMaxScaler()

scaler_bb.fit(X_train[['Middle_BB']].values)  

X_train_df['Upper_BB'] = scaler_bb.transform(X_train[['Upper_BB']].values).flatten()
X_train_df['Middle_BB'] = scaler_bb.transform(X_train[['Middle_BB']].values).flatten()
X_train_df['Lower_BB'] = scaler_bb.transform(X_train[['Lower_BB']].values).flatten()

X_test_df['Upper_BB'] = scaler_bb.transform(X_test[['Upper_BB']].values).flatten()
X_test_df['Middle_BB'] = scaler_bb.transform(X_test[['Middle_BB']].values).flatten()
X_test_df['Lower_BB'] = scaler_bb.transform(X_test[['Lower_BB']].values).flatten()

# MACD
scaler_macd = MinMaxScaler()
scaler_macd.fit(X_train[["MACD"]].values)

X_train_df['MACD'] = scaler_macd.transform(X_train[['MACD']].values).flatten()
X_train_df['MACD_signal'] = scaler_macd.transform(X_train[['MACD_signal']].values).flatten()

X_test_df['MACD'] = scaler_macd.transform(X_test[['MACD']].values).flatten()
X_test_df['MACD_signal'] = scaler_macd.transform(X_test[['MACD_signal']].values).flatten()

scaler_sma = MinMaxScaler()
scaler_sma.fit(X_train[["SMA_SHORT"]].values)

X_train_df['SMA_SHORT'] = scaler_sma.transform(X_train[['SMA_SHORT']].values).flatten()
X_train_df['SMA_LONG'] = scaler_sma.transform(X_train[['SMA_LONG']].values).flatten()

X_test_df['SMA_SHORT'] = scaler_sma.transform(X_test[['SMA_SHORT']].values).flatten()
X_test_df['SMA_LONG'] = scaler_sma.transform(X_test[['SMA_LONG']].values).flatten()

scaler_rsi = MinMaxScaler()
scaler_rsi.fit(X_train[["RSI"]].values)

X_train_df['RSI'] = scaler_rsi.transform(X_train[['RSI']].values).flatten()
X_test_df['RSI'] = scaler_rsi.transform(X_test[['RSI']].values).flatten()


X_test_df

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close
1180,0.499847,0.411144,0.517394,0.387564,0.487569,0.871419,0.791684,0.711949,0.791684,0.795195,0.833706,0.876575,0.853920,0.764804,0.775662,0.793797
1181,0.523637,0.432238,0.527281,0.393264,0.448749,0.874356,0.793223,0.712091,0.793223,0.794354,0.835055,0.895160,0.867699,0.724522,0.788577,0.797684
1182,0.523008,0.448973,0.456307,0.359880,0.379729,0.873634,0.792793,0.711952,0.792793,0.792167,0.835217,0.907286,0.887937,0.685712,0.778442,0.775317
1183,0.534244,0.464868,0.497611,0.382098,0.444492,0.871829,0.792104,0.712380,0.792104,0.791675,0.836117,0.913913,0.901248,0.661729,0.787383,0.790114
1184,0.547371,0.480511,0.514480,0.391192,0.467132,0.873423,0.792814,0.712206,0.792814,0.791100,0.837282,0.923475,0.911343,0.623617,0.797445,0.796184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1470,0.331987,0.360467,0.219194,0.330143,0.438975,1.179515,1.104494,1.029472,1.104494,1.108087,1.152418,0.696649,0.699736,0.263755,0.912203,1.018693
1471,0.358007,0.354748,0.456177,0.434230,0.530971,1.173066,1.101858,1.030651,1.101858,1.109861,1.152792,0.728273,0.697238,0.316961,0.925666,1.059493
1472,0.396416,0.358741,0.541662,0.480565,0.534505,1.166205,1.099905,1.033606,1.099905,1.111685,1.154161,0.814261,0.731129,0.316623,0.937531,1.079584
1473,0.440664,0.371807,0.601114,0.515680,0.555950,1.164928,1.099564,1.034201,1.099564,1.113472,1.156274,0.926300,0.813002,0.323439,0.947909,1.095561


In [29]:
correlation_matrix = X_train_df.corr()

# Create the heatmap with text
fig = go.Figure(data=go.Heatmap(
                    z=correlation_matrix,
                    x=correlation_matrix.columns,
                    y=correlation_matrix.columns,
                    colorscale='Viridis',
                    text=correlation_matrix.round(2).values,  # Rounded values for display
                    texttemplate="%{text}",
                    textfont={"size":9}  # Adjust text size if necessary
                    ))

# Update the layout
fig.update_layout(
    title='Correlation Matrix',
    xaxis_title="Variables",
    yaxis_title="Variables",
    xaxis=dict(side='bottom'),
    yaxis=dict(autorange='reversed'),
    width=1000,  # or any width you desire
    height=1000,  # or any height you desire
)

# Show the figure
pyo.iplot(fig)

In [30]:
X_train_tensor = torch.tensor(X_train_df.to_numpy(), dtype=torch.float, device=device)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float, device=device)

X_test_tensor = torch.tensor(X_test_df.to_numpy(), dtype=torch.float, device=device)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float, device=device)

train_data = RollingWindowDataset(X_train_tensor, y_train_tensor, window_size=time_step, device=device)
test_data = RollingWindowDataset(X_test_tensor, y_test_tensor, window_size=time_step, device=device)

print(test_data.__getitem__(0)[1])
print(test_data.__getitem__(0)[0])


tensor(2., device='cuda:0')
tensor([[0.4998, 0.4111, 0.5174, 0.3876, 0.4876, 0.8714, 0.7917, 0.7119, 0.7917,
         0.7952, 0.8337, 0.8766, 0.8539, 0.7648, 0.7757, 0.7938],
        [0.5236, 0.4322, 0.5273, 0.3933, 0.4487, 0.8744, 0.7932, 0.7121, 0.7932,
         0.7944, 0.8351, 0.8952, 0.8677, 0.7245, 0.7886, 0.7977],
        [0.5230, 0.4490, 0.4563, 0.3599, 0.3797, 0.8736, 0.7928, 0.7120, 0.7928,
         0.7922, 0.8352, 0.9073, 0.8879, 0.6857, 0.7784, 0.7753],
        [0.5342, 0.4649, 0.4976, 0.3821, 0.4445, 0.8718, 0.7921, 0.7124, 0.7921,
         0.7917, 0.8361, 0.9139, 0.9012, 0.6617, 0.7874, 0.7901],
        [0.5474, 0.4805, 0.5145, 0.3912, 0.4671, 0.8734, 0.7928, 0.7122, 0.7928,
         0.7911, 0.8373, 0.9235, 0.9113, 0.6236, 0.7974, 0.7962],
        [0.5399, 0.4914, 0.4471, 0.3605, 0.4592, 0.8745, 0.7941, 0.7137, 0.7941,
         0.7905, 0.8374, 0.9237, 0.9172, 0.5981, 0.7914, 0.7760],
        [0.5112, 0.4936, 0.3677, 0.3216, 0.4080, 0.8655, 0.7900, 0.7146, 0.7900,
         

# Vanilla LSTM

In [31]:
class VanillaLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, layer_size, output_dim, dropout_prob):
        super(VanillaLSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.layer_size = layer_size

        self.lstm = nn.LSTM(input_size = input_dim, hidden_size = hidden_size, num_layers=layer_size,
                            dropout=(dropout_prob if layer_size > 1 else 0), batch_first=True)
                            
        self.dropout = nn.Dropout(dropout_prob)
        
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        # Initializing hidden state
        h0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Initialize cell state
        c0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))

        out = self.dropout(out[:, -1, :])  # Add dropout

        out = self.fc(out)

        return out

# Stacked LSTM

In [32]:
class StackedLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, layer_size, output_dim, dropout_prob, stateful):
        super(StackedLSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.layer_size = layer_size
        self.stateful = stateful
        
        self.hn = None
        self.cn = None

        self.lstm = nn.LSTM(input_size = input_dim, hidden_size = hidden_size, num_layers=self.layer_size,
                            dropout=(dropout_prob if self.layer_size > 1 else 0), batch_first=True)
                            
        self.dropout = nn.Dropout(dropout_prob)
        
        self.fc = nn.Linear(hidden_size, output_dim)

    def reset_hidden_state(self, batch_size=None):
        if batch_size is None or not self.stateful:
            # Reset hidden and cell states to None if batch_size is not provided or if the model is not stateful
            self.hn = None
            self.cn = None
        else:
            # Resize hidden and cell states to match current batch size, preserving the state as much as possible
            self.hn = self._resize_state(self.hn, batch_size)
            self.cn = self._resize_state(self.cn, batch_size)


    def _resize_state(self, state, batch_size):
        if state is None:
            # If state is not initialized, initialize with zeros
            return torch.zeros(self.layer_size, batch_size, self.hidden_size).to(device)
        elif batch_size < state.size(1):
            # If batch size is smaller, truncate the state
            return state[:, :batch_size, :].contiguous()
        elif batch_size > state.size(1):
            # If batch size is larger, pad the state with zeros
            padding_size = batch_size - state.size(1)
            padding = torch.zeros(self.layer_size, padding_size, self.hidden_size).to(device)
            return torch.cat([state, padding], dim=1)
        

    def forward(self, x):
        current_batch_size = x.size(0)

        # Check and adjust the hidden and cell states
        if not self.stateful or self.hn is None or self.hn.size(1) != current_batch_size:
            self.reset_hidden_state(current_batch_size)
        else:
            # Detach the hidden states from the graph to avoid backpropagating through the entire sequence history
            self.hn = self.hn.detach()
            self.cn = self.cn.detach()

        # Ensure that hn and cn are not None and have the correct shape
        h0 = self.hn if self.hn is not None else torch.zeros(self.layer_size, current_batch_size, self.hidden_size).to(device)
        c0 = self.cn if self.cn is not None else torch.zeros(self.layer_size, current_batch_size, self.hidden_size).to(device)

        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0, c0))

        # Update hidden states if stateful
        if self.stateful:
            self.hn, self.cn = hn.detach(), cn.detach()

        # Process the output of the last time step
        out = self.dropout(out[:, -1, :])  # Add dropout
        out = self.fc(out)  # Fully connected layer

        return out

# Bidirectional LSTM

# 1D CNN-LSTM

In [33]:
class OneDimCNNLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, layer_size, output_dim, dropout_prob, conv_channels, kernel_size, pool_size, stride):
        super(OneDimCNNLSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.layer_size = layer_size

        # Convolutional Layer
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=conv_channels, kernel_size=kernel_size)
        self.relu1 = nn.ReLU()
        self.maxpool1 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_size)

        # self.conv2 = nn.Conv1d(in_channels=conv_channels, out_channels=conv_channels*2, kernel_size=kernel_size)
        # self.relu2 = nn.ReLU()
        # self.maxpool2 = nn.MaxPool1d(kernel_size=2, stride=2)


        self.lstm = nn.LSTM(input_size = conv_channels, hidden_size = hidden_size, num_layers=layer_size,
                            dropout=(dropout_prob if layer_size > 1 else 0), batch_first=True)
                            
        self.dropout = nn.Dropout(dropout_prob)
        
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        # CNN in_dim: (batch_size, in_channels, length)
        x = x.transpose(1, 2)

        # Conv layer forward propagate
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.maxpool1(x)

        # x = self.conv2(x)
        # x = self.relu2(x)
        # x = self.maxpool2(x)

        # LSTM in_dim: (batch_size, seq_len, input_size)
        x = x.transpose(1, 2)

        assert x.size(-1) == self.lstm.input_size, f"Mismatch in LSTM input size. Expected: {self.lstm.input_size}, Got: {x.size(-1)}"

        # Initializing hidden state
        h0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Initialize cell state
        c0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))

        out = self.dropout(out[:, -1, :])  # Add dropout

        out = self.fc(out)

        return out

In [34]:
class ModelActioner:
    
    def __init__(self, train_data, test_data, device, model_type):
        self.train_data = train_data
        self.test_data = test_data
        self.device = device
        self.model_type = model_type
        self.model = None
        self.optimizer = None
        self.criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    
    def train_validate(self, config, trial):

        if self.model_type == "Vanilla LSTM":
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = 1
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]

            self.model = VanillaLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=5).to(self.device)
            
        elif self.model_type == "Stacked LSTM":
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = config["num_layers"]
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]
            
            self.model = StackedLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, stateful=False, output_dim=5).to(self.device)

        elif self.model_type == "1D CNN-LSTM":
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = config["num_layers"]
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]
            kernel_size = config["kernel_size"]
            conv_channels = config["conv_channels"]
            pool_size = config["pool_size"]
            stride = config["stride"]

            self.model = OneDimCNNLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=5, conv_channels=conv_channels, kernel_size=kernel_size, pool_size=pool_size, stride=stride).to(self.device)

        n_splits = 5
        tscv = TimeSeriesSplit(n_splits=n_splits)

        val_losses = []

        for fold, (train_ids, val_ids) in enumerate(tscv.split(self.train_data)):
            print(f'Starting fold {fold+1}:')

            self.optimizer = optim.Adam(self.model.parameters(), lr = learning_rate, weight_decay=weight_decay)

            scheduler = ReduceLROnPlateau(self.optimizer, patience=lr_step_size, factor=gamma, mode="min", verbose=True) 

            train_subset = Subset(self.train_data, train_ids)
            val_subset = Subset(self.train_data, val_ids)
            
            # Creating data loader
            train_loader = DataLoader(dataset=train_subset, batch_size=batch_size, shuffle=False)
            val_loader = DataLoader(dataset=val_subset, batch_size=batch_size, shuffle=False)

            # Training Loop
            for epoch in range(epochs):
                print('epochs {}/{}'.format(epoch+1,epochs))

                running_loss = 0.0
                total_sample_train = 0

                self.model.train()

                for batch_idx, (data, target) in enumerate(train_loader):
                    data, target = data.to(self.device), target.to(self.device)
                    target = target.long()


                    self.optimizer.zero_grad()
                    preds = self.model(data)

                    loss = self.criterion(preds, target)
                    #loss = loss.mean()
                    loss.backward()
                    self.optimizer.step() # Update model params

                    running_loss += loss.item() * data.size(0)
                    total_sample_train += data.size(0)

                train_loss = running_loss/total_sample_train
                #print(f"train loss: {train_loss}")

                self.model.eval()
                val_running_loss = 0.0
                total_sample_val = 0

                with torch.no_grad():

                    for batch_idx, (data, target) in enumerate(val_loader):
                        data, target = data.to(self.device), target.to(self.device)
                        target = target.long()

                        preds = self.model(data)
                        loss = self.criterion(preds, target)
                        #loss = loss.mean()

                        val_running_loss += loss.item() * data.size(0)
                        total_sample_val += data.size(0)
                
                if trial.should_prune():
                    raise optuna.TrialPruned()
                
                val_loss = val_running_loss/total_sample_val
                val_losses.append(val_loss)
                scheduler.step(val_loss)
                
                unique_step = fold * epochs + epoch
                trial.report(val_loss, unique_step)

                current_lr = self.optimizer.param_groups[0]['lr']

                print(f'Current Learning Rate: {current_lr}')
                print(f"train_loss: {train_loss}, val_loss: {val_loss}")
                
        mean_val_loss = np.mean(val_losses)
        print(f"Mean validation loss: {mean_val_loss}")
        return mean_val_loss
    
                    
    def train(self, config):
        if self.model_type == "Vanilla LSTM":

            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = 1
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]

            self.model = VanillaLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=5).to(self.device)
            
        elif self.model_type == "Stacked LSTM":
            
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = config["num_layers"]
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]
            
            self.model = StackedLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, stateful=False, output_dim=5).to(self.device)

        elif self.model_type == "1D CNN-LSTM":
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = config["num_layers"]
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]
            kernel_size = config["kernel_size"]
            conv_channels = config["conv_channels"]
            pool_size = config["pool_size"]
            stride = config["stride"]

            self.model = OneDimCNNLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=5, conv_channels=conv_channels, kernel_size=kernel_size, pool_size=pool_size, stride=stride).to(self.device)

        # Update optimizer with updated lr
        self.optimizer = optim.Adam(self.model.parameters(), lr = learning_rate, weight_decay=weight_decay)

        # Creating data loader
        train_loader = DataLoader(dataset=self.train_data, batch_size=batch_size, shuffle=False)

        scheduler = ReduceLROnPlateau(self.optimizer, patience=lr_step_size, factor=gamma, mode="min", verbose=True)  

        # Training Loop
        for epoch in range(epochs):
            print('epochs {}/{}'.format(epoch+1,epochs))

            running_loss = 0.0
            total_sample_train = 0

            self.model.train()

            for batch_idx, (data, target) in enumerate(train_loader):

                data, target = data.to(self.device), target.to(self.device)
                target = target.long()  

                self.optimizer.zero_grad()
                preds = self.model(data)

                loss = self.criterion(preds, target)
                #loss = loss.mean()
                loss.backward()
                self.optimizer.step() # Update model params

                running_loss += loss.item() * data.size(0)
                total_sample_train += data.size(0)

            train_loss = running_loss/total_sample_train
            #print(f"train loss: {train_loss}")
            scheduler.step(train_loss)
            current_lr = self.optimizer.param_groups[0]['lr']

            print(f'Current Learning Rate: {current_lr}')
            print(f"train_loss: {train_loss}")
        
        return self.model
            
    
    def test(self, config):
        batch_size = config["batch_size"]
        all_preds = []

        test_loader = DataLoader(dataset=self.test_data, batch_size=batch_size, shuffle=False)

        running_loss = .0
        total_sample = 0

        self.model.eval()

        with torch.no_grad():

            for batch_idx, (data, target) in enumerate(test_loader):

                data, target = data.to(self.device), target.to(self.device)
                target = target.long()
                
                preds = self.model(data)
                loss = self.criterion(preds, target)

                running_loss += loss.item() * data.size(0)
                total_sample += data.size(0)

                _, predicted_classes = torch.max(preds, 1)
                all_preds.extend(predicted_classes.cpu().numpy())

            test_loss = running_loss/total_sample
            print(f"test_loss: {test_loss}")

        return all_preds
    


In [35]:
model_type = "1D CNN-LSTM"

def objective(trial):
    if model_type == "Vanilla LSTM":
        config = {
            "batch_size": trial.suggest_int("batch_size", 32, 128, log=True),
            "epochs": trial.suggest_int("epochs", 50, 150),
            "hidden_size": trial.suggest_int("hidden_size", 10, 100),
            "learning_rate": trial.suggest_float("learning_rate", 1e-6, 1e-1, log=True),
            "dropout_prob": trial.suggest_float("dropout_prob", 0.0, 0.15),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
            "lr_step_size": trial.suggest_int("lr_step_size", 10, 100), 
            "gamma": trial.suggest_float("gamma", 0.1, 0.9)
        }

    elif model_type == "Stacked LSTM":
        config = {
            "batch_size": trial.suggest_int("batch_size", 32, 128, log=True),
            "epochs": trial.suggest_int("epochs", 50, 150),
            "hidden_size": trial.suggest_int("hidden_size", 10, 100),
            "learning_rate": trial.suggest_float("learning_rate", 1e-6, 1e-1, log=True),
            "dropout_prob": trial.suggest_float("dropout_prob", 0.0, 0.2),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
            "lr_step_size": trial.suggest_int("lr_step_size", 5, 100), 
            "gamma": trial.suggest_float("gamma", 0.1, 0.9),
            "num_layers": trial.suggest_int("num_layers", 1, 5)
        }

    elif model_type == "1D CNN-LSTM":
        config = {
            "batch_size": trial.suggest_int("batch_size", 32, 128, log=True),
            "epochs": trial.suggest_int("epochs", 50, 150),
            "hidden_size": trial.suggest_int("hidden_size", 10, 100),
            "learning_rate": trial.suggest_float("learning_rate", 1e-6, 1e-1, log=True),
            "dropout_prob": trial.suggest_float("dropout_prob", 0.0, 0.2),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
            "lr_step_size": trial.suggest_int("lr_step_size", 5, 100), 
            "gamma": trial.suggest_float("gamma", 0.1, 0.9),
            "conv_channels": trial.suggest_int("conv_channels", 16, 128),
            "kernel_size": trial.suggest_int("kernel_size", 2, 6),
            "num_layers": trial.suggest_int("num_layers", 1, 5),
            "pool_size": trial.suggest_int("pool_size", 2, 5),
            "stride": trial.suggest_int("stride", 1, 4)
        }

    trainer = ModelActioner(train_data, test_data, device, model_type)

    val_loss = trainer.train_validate(config, trial)

    return val_loss

In [36]:
study_name = "Vanilla-LSTM-Tunner"
storage_url = "sqlite:///db.sqlite3"

optuna.delete_study(study_name=study_name, storage= storage_url)

study = optuna.create_study(direction='minimize', 
                            storage=storage_url, 
                            sampler=TPESampler(),
                            pruner=optuna.pruners.HyperbandPruner(
                            min_resource=30,  
                            max_resource=150,  
                            reduction_factor=3,  
                            ),
                            study_name=study_name,
                            load_if_exists=False)

pbar = tqdm(total=15, desc='Optimizing', unit='trial')

def callback(study, trial):
    # Update the progress bar
    pbar.update(1)
    # Optionally, you can include additional information in the progress bar
    pbar.set_postfix_str(f"Best Value: {study.best_value:.4f}")


study.optimize(objective, n_trials=15, callbacks=[callback])
pbar.close()

# Best hyperparameters
print('Number of finished trials:', len(study.trials))
print('Best trial:')
trial = study.best_trial

print('Value:', trial.value)
print('Params:')
for key, value in trial.params.items():
    print(f'{key}: {value}')

[I 2024-01-28 14:56:20,546] A new study created in RDB with name: Vanilla-LSTM-Tunner


Optimizing:   0%|          | 0/15 [00:00<?, ?trial/s]

Starting fold 1:
epochs 1/70
Current Learning Rate: 4.05999705912671e-06
train_loss: 1.6254671749644254, val_loss: 1.5595686303244696
epochs 2/70
Current Learning Rate: 4.05999705912671e-06
train_loss: 1.6253889218674904, val_loss: 1.5595051712459989
epochs 3/70
Current Learning Rate: 4.05999705912671e-06
train_loss: 1.6257305550949737, val_loss: 1.5594481892055936
epochs 4/70
Current Learning Rate: 4.05999705912671e-06
train_loss: 1.625314801775348, val_loss: 1.559394015206231
epochs 5/70
Current Learning Rate: 4.05999705912671e-06
train_loss: 1.6272562711026657, val_loss: 1.5593411127726238
epochs 6/70
Current Learning Rate: 4.05999705912671e-06
train_loss: 1.624392825895579, val_loss: 1.5592881970935397
epochs 7/70
Current Learning Rate: 4.05999705912671e-06
train_loss: 1.6254834024069822, val_loss: 1.5592350429958768
epochs 8/70
Current Learning Rate: 4.05999705912671e-06
train_loss: 1.622115182626934, val_loss: 1.559181849161784
epochs 9/70
Current Learning Rate: 4.05999705912671e

[I 2024-01-28 14:56:57,793] Trial 0 finished with value: 1.5985600346232218 and parameters: {'batch_size': 42, 'epochs': 70, 'hidden_size': 30, 'learning_rate': 4.05999705912671e-06, 'dropout_prob': 0.005070853974255618, 'weight_decay': 0.004294981668669153, 'lr_step_size': 58, 'gamma': 0.15438732294376198, 'conv_channels': 97, 'kernel_size': 5, 'num_layers': 2, 'pool_size': 5, 'stride': 4}. Best is trial 0 with value: 1.5985600346232218.


Current Learning Rate: 4.05999705912671e-06
train_loss: 1.5526597868678689, val_loss: 1.64226410124037
epochs 70/70
Current Learning Rate: 4.05999705912671e-06
train_loss: 1.5529529964030104, val_loss: 1.6417806545893352
Mean validation loss: 1.5985600346232218
Starting fold 1:
epochs 1/82
Current Learning Rate: 3.572889522949767e-05
train_loss: 1.5976275486471765, val_loss: 1.5569378295272747
epochs 2/82
Current Learning Rate: 3.572889522949767e-05
train_loss: 1.6002826160161283, val_loss: 1.5567026384293088
epochs 3/82
Current Learning Rate: 3.572889522949767e-05
train_loss: 1.6036860599567753, val_loss: 1.5565177870805933
epochs 4/82
Current Learning Rate: 3.572889522949767e-05
train_loss: 1.5990099725922988, val_loss: 1.5564243541192757
epochs 5/82
Current Learning Rate: 3.572889522949767e-05
train_loss: 1.5924856550406412, val_loss: 1.5563999332448162
epochs 6/82
Current Learning Rate: 3.572889522949767e-05
train_loss: 1.6051772171290133, val_loss: 1.5563785998278825
epochs 7/82
C

...

Current Learning Rate: 7.010654015664084e-06
train_loss: 1.619231337028024, val_loss: 1.5722563443360504
epochs 89/95
Current Learning Rate: 7.010654015664084e-06
train_loss: 1.6189627347816347, val_loss: 1.5722673014988975
epochs 90/95
Current Learning Rate: 7.010654015664084e-06
train_loss: 1.6190840012115957, val_loss: 1.572279241349962
epochs 91/95
Current Learning Rate: 7.010654015664084e-06
train_loss: 1.6190967815708739, val_loss: 1.5722916315472315
epochs 92/95
Starting fold 1:
epochs 1/52
Current Learning Rate: 6.426489847556882e-06
train_loss: 1.622257974135314, val_loss: 1.6073909051834592
epochs 2/52
Current Learning Rate: 6.426489847556882e-06
train_loss: 1.6218803652918152, val_loss: 1.6073763805722434
epochs 3/52
Current Learning Rate: 6.426489847556882e-06
train_loss: 1.6242647907496748, val_loss: 1.6073644255834914
epochs 4/52
Current Learning Rate: 6.426489847556882e-06
train_loss: 1.6236064128226635, val_loss: 1.6073522643437461
epochs 5/52
Current Learning Rate: 6.4

[I 2024-01-28 14:59:01,645] Trial 13 pruned. 


Starting fold 1:
epochs 1/77
Current Learning Rate: 0.06710940387989861
train_loss: 1.846102340683263, val_loss: 1.998025513830639
epochs 2/77
Current Learning Rate: 0.06710940387989861
train_loss: 1.9975821765929616, val_loss: 1.7974305146585696
epochs 3/77
Current Learning Rate: 0.06710940387989861
train_loss: 1.797826235831096, val_loss: 1.5094096042491771
epochs 4/77
Current Learning Rate: 0.06710940387989861
train_loss: 1.8799233598858898, val_loss: 1.481147016797747
epochs 5/77
Current Learning Rate: 0.06710940387989861
train_loss: 1.668148807206079, val_loss: 1.5377329701469058
epochs 6/77
Current Learning Rate: 0.06710940387989861
train_loss: 1.5900586937110461, val_loss: 1.535312536532286
epochs 7/77
Current Learning Rate: 0.06710940387989861
train_loss: 1.6594697499150382, val_loss: 1.5205810032193623
epochs 8/77
Current Learning Rate: 0.06710940387989861
train_loss: 1.6609369984471984, val_loss: 1.5072459550130934
epochs 9/77
Current Learning Rate: 0.06710940387989861
train_

[I 2024-01-28 14:59:03,662] Trial 14 pruned. 


Current Learning Rate: 0.06710940387989861
train_loss: 1.6981046293418445, val_loss: 1.544945761009499
epochs 31/77
Current Learning Rate: 0.06710940387989861
train_loss: 1.6212417835964583, val_loss: 1.5726035435994465
epochs 32/77
Number of finished trials: 15
Best trial:
Value: 1.4652380997511103
Params:
batch_size: 62
epochs: 103
hidden_size: 20
learning_rate: 0.00013635815806655385
dropout_prob: 0.16281589392460533
weight_decay: 3.898854912850674e-06
lr_step_size: 58
gamma: 0.7120345424530452
conv_channels: 123
kernel_size: 2
num_layers: 2
pool_size: 2
stride: 2


In [37]:
trainer = ModelActioner(train_data, test_data, device, model_type)
model = trainer.train(trial.params)

epochs 1/103
Current Learning Rate: 0.00013635815806655385
train_loss: 1.620214872586895
epochs 2/103
Current Learning Rate: 0.00013635815806655385
train_loss: 1.6113953111876904
epochs 3/103
Current Learning Rate: 0.00013635815806655385
train_loss: 1.613412884129605
epochs 4/103
Current Learning Rate: 0.00013635815806655385
train_loss: 1.6046101081539208
epochs 5/103
Current Learning Rate: 0.00013635815806655385
train_loss: 1.5939168470426344
epochs 6/103
Current Learning Rate: 0.00013635815806655385
train_loss: 1.5845008790493011
epochs 7/103
Current Learning Rate: 0.00013635815806655385
train_loss: 1.576847416204466
epochs 8/103
Current Learning Rate: 0.00013635815806655385
train_loss: 1.5705800511887376
epochs 9/103
Current Learning Rate: 0.00013635815806655385
train_loss: 1.5695385040951446
epochs 10/103
Current Learning Rate: 0.00013635815806655385
train_loss: 1.5594417312195603
epochs 11/103
Current Learning Rate: 0.00013635815806655385
train_loss: 1.5505509569611349
epochs 12/1

In [38]:
preds = trainer.test(trial.params)
y_true = y_test[time_step:]

accuracy = accuracy_score(y_true, preds)
print(f'Accuracy: {accuracy * 100:.2f}%')

# Precision
precision = precision_score(y_true, preds, average='weighted', zero_division=0)
print(f'Precision: {precision:.4f}')

# Recall
recall = recall_score(y_true, preds, average='weighted')
print(f'Recall: {recall:.4f}')

# F1 Score
f1 = f1_score(y_true, preds, average='weighted')
print(f'F1 Score: {f1:.4f}')

test_loss: 1.968434199393983
Accuracy: 43.43%
Precision: 0.3367
Recall: 0.4343
F1 Score: 0.3341


In [39]:
unique_elements = set(preds)
print(unique_elements)

{0, 2, 3}


In [40]:
signals = pd.DataFrame(preds, columns=['Signal'])
signals

,Signal
0,2
1,2
2,2
3,2
4,2
...,...
246,3
247,0
248,0
249,0


In [41]:
signals["Signal"].value_counts()

Signal
3    165
2     62
0     24
Name: count, dtype: int64

In [42]:
signals["Date"] = date_test
signals

,Signal,Date
0,2,2023-01-23
1,2,2023-01-24
2,2,2023-01-25
3,2,2023-01-26
4,2,2023-01-27
...,...,...
246,3,2024-01-16
247,0,2024-01-17
248,0,2024-01-18
249,0,2024-01-19


In [43]:
stock_price = stock_data["Adj Close"].iloc[test_index:]
stock_price=stock_price.reset_index()
stock_price=stock_price.drop(columns=["index"])
stock_price

,Adj Close
0,140.325638
1,141.737747
2,141.071472
3,143.159805
4,145.118851
...,...
247,182.679993
248,188.630005
249,191.559998
250,193.889999


In [44]:
date_test["Date"] = date_test["Date"].dt.strftime('%Y-%m-%d')
date_test

,Date
0,2023-01-23
1,2023-01-24
2,2023-01-25
3,2023-01-26
4,2023-01-27
...,...
247,2024-01-17
248,2024-01-18
249,2024-01-19
250,2024-01-22


In [45]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.array(date_test).flatten(), y=stock_data["Adj Close"].iloc[test_index:], mode='lines', name='TSLA Stock Price'))

# Buy and sell signals
strong_sell_signals = signals[signals['Signal'] == 1]
sell_signals = signals[signals['Signal'] == 2]
buy_signals = signals[signals['Signal'] == 3]
strong_buy_signals = signals[signals['Signal'] == 4]


# Ensure that buy and sell prices are aligned with the signals by matching on the 'Date' column
buy_prices = stock_data[stock_data['Date'].isin(buy_signals['Date'])]["Adj Close"]
sell_prices = stock_data[stock_data['Date'].isin(sell_signals['Date'])]["Adj Close"]
strong_buy_prices = stock_data[stock_data['Date'].isin(strong_buy_signals['Date'])]["Adj Close"]
strong_sell_prices = stock_data[stock_data['Date'].isin(strong_sell_signals['Date'])]["Adj Close"]

# Plot buy signals
fig.add_trace(go.Scatter(
    x=buy_signals['Date'], 
    y=buy_prices, 
    mode='markers', 
    name='Buy', 
    marker=dict(color='lime', size=10, symbol='triangle-up')
))

fig.add_trace(go.Scatter(
    x=strong_buy_signals['Date'], 
    y=strong_buy_prices, 
    mode='markers', 
    name='Buy', 
    marker=dict(color='green', size=10, symbol='triangle-up')
))

# Plot sell signals
fig.add_trace(go.Scatter(
    x=sell_signals['Date'], 
    y=sell_prices, 
    mode='markers', 
    name='Sell', 
    marker=dict(color='orange', size=10, symbol='triangle-down')
))

fig.add_trace(go.Scatter(
    x=strong_sell_signals['Date'], 
    y=strong_sell_prices, 
    mode='markers', 
    name='Sell', 
    marker=dict(color='red', size=10, symbol='triangle-down')
))

# Update layout
fig.update_layout(
    title='Stock Price with Buy and Sell Signals',
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False,
    height = 700,
    width=1280
)

# Show the plot
pyo.iplot(fig)

/home/arda/anaconda3/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



In [46]:
price_signal = stock_data[stock_data['Date'].isin(signals['Date'])]["Adj Close"]
price_signal = price_signal.reset_index()
price_signal = price_signal.drop(columns=["index"])
result = pd.concat([price_signal,signals], axis=1)
result

,Adj Close,Signal,Date
0,140.325638,2,2023-01-23
1,141.737747,2,2023-01-24
2,141.071472,2,2023-01-25
3,143.159805,2,2023-01-26
4,145.118851,2,2023-01-27
...,...,...,...
246,183.630005,3,2024-01-16
247,182.679993,0,2024-01-17
248,188.630005,0,2024-01-18
249,191.559998,0,2024-01-19


In [47]:
price_signal

,Adj Close
0,140.325638
1,141.737747
2,141.071472
3,143.159805
4,145.118851
...,...
246,183.630005
247,182.679993
248,188.630005
249,191.559998


In [48]:
sell_signals

,Signal,Date
0,2,2023-01-23
1,2,2023-01-24
2,2,2023-01-25
3,2,2023-01-26
4,2,2023-01-27
...,...,...
197,2,2023-11-02
198,2,2023-11-03
199,2,2023-11-06
200,2,2023-11-07


In [49]:
buy_prices

1284    151.075577
1285    150.031387
1286    150.399902
1287    153.228439
1288    152.581055
           ...    
1514    185.139999
1515    186.190002
1516    185.589996
1517    185.919998
1518    183.630005
Name: Adj Close, Length: 165, dtype: float64